In [ ]:
"""
https://github.com/ducphat1509-beep/artificial-intelligence
"""

In [ ]:
"""
Buổi 4
Nguyễn Đức Phát - 24110296
"""

In [1]:
# model-based-vacuumAgent

import tkinter as tk
import random
import time

# =========================
# CONFIG
# =========================
SIZE = 4
CELL_SIZE = 100
DELAY = 500            # milliseconds

DIRTY = 1
CLEAN = 0

# =========================
# MODEL-BASED VACUUM AGENT
# =========================
class VacuumAgent:

    def __init__(self, env):

        self.env = env

        self.x = 0
        self.y = 0

        self.visited_cells = set() # To track visited cells
        self.path_taken = [] # To track the path for potential backtracking/logic

        self.visited_cells.add((self.x, self.y))
        self.path_taken.append((self.x, self.y))

    # =========================
    # SENSOR
    # =========================
    def perceive(self):
        return self.env.grid[self.x][self.y]

    # =========================
    # CHECK POSSIBLE MOVES
    # =========================
    def get_possible_moves(self):
        moves = []
        if self.x > 0: moves.append(("UP", self.x - 1, self.y))
        if self.x < SIZE - 1: moves.append(("DOWN", self.x + 1, self.y))
        if self.y > 0: moves.append(("LEFT", self.x, self.y - 1))
        if self.y < SIZE - 1: moves.append(("RIGHT", self.x, self.y + 1))
        return moves

    # =========================
    # ACTUATOR + RULES
    # =========================
    def act(self):
        current_state = self.perceive()

        # RULE 1: If current cell is dirty, suck it
        if current_state == DIRTY:
            self.env.grid[self.x][self.y] = CLEAN
            print(f"SUCK at ({self.x}, {self.y})")
            return

        # RULE 2: If clean, decide next move based on visited cells
        # Get all physically possible moves
        all_possible_moves = self.get_possible_moves()

        # Filter out moves that lead to already visited cells
        unvisited_moves = [
            (action, next_x, next_y) for action, next_x, next_y in all_possible_moves
            if (next_x, next_y) not in self.visited_cells
        ]

        chosen_action = None
        if unvisited_moves:
            # Prioritize moving to an unvisited cell
            chosen_action = random.choice(unvisited_moves)
        else:
            # If all adjacent cells are visited, then pick a random possible move,
            # even if visited, to avoid getting completely stuck. This ensures
            # the agent can eventually reach all dirty cells.
            if all_possible_moves:
                chosen_action = random.choice(all_possible_moves)
            else:
                print("Agent is stuck! No possible moves.")
                return

        if chosen_action:
            action, next_x, next_y = chosen_action
            self.move(action, next_x, next_y)

    # =========================
    # MOVE
    # =========================
    def move(self, action, next_x, next_y):
        self.x = next_x
        self.y = next_y

        self.visited_cells.add((self.x, self.y))
        self.path_taken.append((self.x, self.y)) # Keep track of the path for debugging/advanced logic

        print(f"MOVE {action} -> ({self.x}, {self.y})")

    # =========================
    # CHECK ALL CLEAN
    # =========================
    def all_clean(self):
        for row in self.env.grid:
            for cell in row:
                if cell == DIRTY:
                    return False
        return True


# =========================
# ENVIRONMENT
# =========================
class Environment:

    def __init__(self):

        # Tạo ma trận random sạch/bẩn
        self.grid = [
            [random.choice([DIRTY, CLEAN]) for _ in range(SIZE)]
            for _ in range(SIZE)
        ]


# =========================
# GUI
# =========================
class VacuumGUI:

    def __init__(self, root):

        self.root = root
        self.root.title("Model-Based Vacuum Agent")

        self.canvas = tk.Canvas(
            root,
            width=SIZE * CELL_SIZE,
            height=SIZE * CELL_SIZE
        )

        self.canvas.pack()

        self.env = Environment()
        self.agent = VacuumAgent(self.env)

        self.draw_grid()

        # Start simulation
        self.root.after(DELAY, self.update)

    def draw_grid(self):

        self.canvas.delete("all")

        for i in range(SIZE):
            for j in range(SIZE):

                x1 = j * CELL_SIZE
                y1 = i * CELL_SIZE
                x2 = x1 + CELL_SIZE
                y2 = y1 + CELL_SIZE

                cell = self.env.grid[i][j]

                # Dirty = nâu
                if cell == DIRTY:
                    color = "saddle brown"
                else:
                    color = "white"

                self.canvas.create_rectangle(
                    x1, y1, x2, y2,
                    fill=color,
                    outline="black",
                    width=2
                )

                # Hiển thị trạng thái
                text = "DIRTY" if cell == DIRTY else "CLEAN"

                self.canvas.create_text(
                    x1 + CELL_SIZE // 2,
                    y1 + CELL_SIZE // 2,
                    text=text,
                    font=("Arial", 12, "bold")
                )

                # Mark visited cells (optional, for visualization)
                if (i, j) in self.agent.visited_cells:
                    self.canvas.create_oval(
                        x1 + CELL_SIZE // 4,
                        y1 + CELL_SIZE // 4,
                        x2 - CELL_SIZE // 4,
                        y2 - CELL_SIZE // 4,
                        fill="lightgray",
                        outline="gray"
                    )

        # Vẽ robot
        rx = self.agent.y * CELL_SIZE + CELL_SIZE // 2
        ry = self.agent.x * CELL_SIZE + CELL_SIZE // 2

        self.canvas.create_oval(
            rx - 25,
            ry - 25,
            rx + 25,
            ry + 25,
            fill="dodger blue"
        )

        self.canvas.create_text(
            rx,
            ry,
            text="Robot Hút Bụi",
            fill="white",
            font=("Arial", 16, "bold")
        )

    def update(self):

        if self.agent.all_clean():
            print("ALL CLEANED!")

            self.canvas.create_text(
                SIZE * CELL_SIZE // 2,
                SIZE * CELL_SIZE // 2,
                text="DONE",
                fill="green",
                font=("Arial", 30, "bold")
            )

            return

        self.agent.act()

        self.draw_grid()

        self.root.after(DELAY, self.update)


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # This part might cause TclError in environments without a display.
    # Consider handling this for Colab execution.
    try:
        root = tk.Tk()
        app = VacuumGUI(root)
        root.mainloop()
    except tk.TclError as e:
        print(f"Error: {e}. Tkinter applications often require a graphical display. "
              "This code might not run directly in some cloud environments without a display server."
              "The logic of the agent still works in the background.")
        # If GUI fails, we can still simulate the agent's actions in text.
        env = Environment()
        agent = VacuumAgent(env)
        print("\n--- Simulating agent without GUI ---")
        steps = 0
        max_steps = SIZE * SIZE * 5 # Arbitrary limit to prevent infinite loops if agent gets stuck
        while not agent.all_clean() and steps < max_steps:
            agent.act()
            steps += 1
            # Optional: print grid state after each action for text-based simulation
            # for row in agent.env.grid:
            #     print(row)
            # print(f"Agent at: ({agent.x}, {agent.y}), Visited: {agent.visited_cells}")
            # time.sleep(0.1) # small delay for readability
        if agent.all_clean():
            print(f"All cleaned in {steps} steps (text-based simulation).")
        else:
            print(f"Simulation ended after {steps} steps. Not all cells cleaned (text-based simulation).")


MOVE RIGHT -> (0, 1)
MOVE DOWN -> (1, 1)
MOVE DOWN -> (2, 1)
SUCK at (2, 1)
MOVE DOWN -> (3, 1)
SUCK at (3, 1)
MOVE RIGHT -> (3, 2)
SUCK at (3, 2)
MOVE UP -> (2, 2)
SUCK at (2, 2)
MOVE RIGHT -> (2, 3)
SUCK at (2, 3)
MOVE UP -> (1, 3)
MOVE LEFT -> (1, 2)
MOVE UP -> (0, 2)
SUCK at (0, 2)
MOVE RIGHT -> (0, 3)
MOVE DOWN -> (1, 3)
MOVE UP -> (0, 3)
MOVE LEFT -> (0, 2)
MOVE LEFT -> (0, 1)
MOVE DOWN -> (1, 1)
MOVE LEFT -> (1, 0)
MOVE DOWN -> (2, 0)
SUCK at (2, 0)
ALL CLEANED!


In [1]:
# simple-Reflex-vacuumAgent

import tkinter as tk
import random
import time

# =========================
# CONFIG
# =========================
SIZE = 4               # Có thể đổi thành 5,6,7...
CELL_SIZE = 100
DELAY = 500            # milliseconds

DIRTY = 1
CLEAN = 0

# =========================
# SIMPLE REFLEX VACUUM AGENT
# =========================
class VacuumAgent:

    def __init__(self, env):

        self.env = env

        self.x = 0
        self.y = 0

    # =========================
    # SENSOR
    # =========================
    def perceive(self):

        return self.env.grid[self.x][self.y]

    # =========================
    # CHECK POSSIBLE MOVES
    # =========================
    def possible_moves(self):

        moves = []

        # UP
        if self.x > 0:
            moves.append("UP")

        # DOWN
        if self.x < SIZE - 1:
            moves.append("DOWN")

        # LEFT
        if self.y > 0:
            moves.append("LEFT")

        # RIGHT
        if self.y < SIZE - 1:
            moves.append("RIGHT")

        return moves

    # =========================
    # ACTUATOR + RULES
    # =========================
    def act(self):

        current = self.perceive()

        # RULE 1:
        # Nếu ô hiện tại bẩn -> hút
        if current == DIRTY:

            self.env.grid[self.x][self.y] = CLEAN

            print(f"SUCK at ({self.x}, {self.y})")

            return

        # RULE 2:
        # Kiểm tra possible move
        moves = self.possible_moves()

        # RULE 3:
        # Random action
        action = random.choice(moves)

        self.move(action)

    # =========================
    # MOVE
    # =========================
    def move(self, action):

        if action == "UP":
            self.x -= 1

        elif action == "DOWN":
            self.x += 1

        elif action == "LEFT":
            self.y -= 1

        elif action == "RIGHT":
            self.y += 1

        print(f"MOVE {action} -> ({self.x}, {self.y})")

    # =========================
    # CHECK ALL CLEAN
    # =========================
    def all_clean(self):

        for row in self.env.grid:
            for cell in row:
                if cell == DIRTY:
                    return False

        return True


# =========================
# ENVIRONMENT
# =========================
class Environment:

    def __init__(self):

        # Tạo ma trận random sạch/bẩn
        self.grid = [
            [random.choice([DIRTY, CLEAN]) for _ in range(SIZE)]
            for _ in range(SIZE)
        ]


# =========================
# GUI
# =========================
class VacuumGUI:

    def __init__(self, root):

        self.root = root
        self.root.title("Simple Reflex Vacuum Agent")

        self.canvas = tk.Canvas(
            root,
            width=SIZE * CELL_SIZE,
            height=SIZE * CELL_SIZE
        )

        self.canvas.pack()

        self.env = Environment()
        self.agent = VacuumAgent(self.env)

        self.draw_grid()

        # Start simulation
        self.root.after(DELAY, self.update)

    def draw_grid(self):

        self.canvas.delete("all")

        for i in range(SIZE):
            for j in range(SIZE):

                x1 = j * CELL_SIZE
                y1 = i * CELL_SIZE
                x2 = x1 + CELL_SIZE
                y2 = y1 + CELL_SIZE

                cell = self.env.grid[i][j]

                # Dirty = nâu
                if cell == DIRTY:
                    color = "saddle brown"
                else:
                    color = "white"

                self.canvas.create_rectangle(
                    x1, y1, x2, y2,
                    fill=color,
                    outline="black",
                    width=2
                )

                # Hiển thị trạng thái
                text = "DIRTY" if cell == DIRTY else "CLEAN"

                self.canvas.create_text(
                    x1 + CELL_SIZE // 2,
                    y1 + CELL_SIZE // 2,
                    text=text,
                    font=("Arial", 12, "bold")
                )

        # Vẽ robot
        rx = self.agent.y * CELL_SIZE + CELL_SIZE // 2
        ry = self.agent.x * CELL_SIZE + CELL_SIZE // 2

        self.canvas.create_oval(
            rx - 25,
            ry - 25,
            rx + 25,
            ry + 25,
            fill="dodger blue"
        )

        self.canvas.create_text(
            rx,
            ry,
            text="R",
            fill="white",
            font=("Arial", 16, "bold")
        )

    def update(self):

        if self.agent.all_clean():
            print("ALL CLEANED!")

            self.canvas.create_text(
                SIZE * CELL_SIZE // 2,
                SIZE * CELL_SIZE // 2,
                text="DONE",
                fill="green",
                font=("Arial", 30, "bold")
            )

            return

        self.agent.act()

        self.draw_grid()

        self.root.after(DELAY, self.update)


# =========================
# MAIN
# =========================
if __name__ == "__main__":

    root = tk.Tk()

    app = VacuumGUI(root)

    root.mainloop()

MOVE RIGHT -> (0, 1)
SUCK at (0, 1)
MOVE LEFT -> (0, 0)
MOVE DOWN -> (1, 0)
SUCK at (1, 0)
MOVE RIGHT -> (1, 1)


In [ ]:
# Model Based Agent - 8 Puzzle

import tkinter as tk
import random
from collections import deque

# ======================================================
# 8 Puzzle - Model Based Agent + Loop Detection
# ======================================================
# Có GUI
# Có model trạng thái
# Có kiểm tra solvable
# Có phát hiện loop khi AI solve
# Có BFS tìm lời giải
#
# Một đống ô vuông nhưng nhân loại quyết định:
# "Ừ, đây sẽ là bài toán AI kinh điển."
# Không phải chữa bệnh. Không phải cứu đói.
# Là... kéo số trong ma trận.
# ======================================================

SIZE = 3

GOAL = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)


class PuzzleModel:
    """
    MODEL:
    Lưu trạng thái board
    """

    def __init__(self):
        self.state = [
            [1, 2, 3],
            [4, 5, 6],
            [7, 8, 0]
        ]

    def get_state_tuple(self):
        return tuple(tuple(row) for row in self.state)

    def find_empty(self):
        for i in range(SIZE):
            for j in range(SIZE):
                if self.state[i][j] == 0:
                    return i, j

    def move(self, direction):
        r, c = self.find_empty()

        new_state = [row[:] for row in self.state]

        if direction == "up" and r < SIZE - 1:
            new_state[r][c], new_state[r + 1][c] = \
                new_state[r + 1][c], new_state[r][c]

        elif direction == "down" and r > 0:
            new_state[r][c], new_state[r - 1][c] = \
                new_state[r - 1][c], new_state[r][c]

        elif direction == "left" and c < SIZE - 1:
            new_state[r][c], new_state[r][c + 1] = \
                new_state[r][c + 1], new_state[r][c]

        elif direction == "right" and c > 0:
            new_state[r][c], new_state[r][c - 1] = \
                new_state[r][c - 1], new_state[r][c]

        else:
            return False

        self.state = new_state
        return True

    def shuffle(self, times=100):
        directions = ["up", "down", "left", "right"]

        for _ in range(times):
            self.move(random.choice(directions))


# ======================================================
# Kiểm tra solvable
# ======================================================
def count_inversions(flat):
    arr = [x for x in flat if x != 0]

    inv = 0

    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1

    return inv


def is_solvable(state):
    flat = []

    for row in state:
        flat.extend(row)

    inv = count_inversions(flat)

    # 3x3 solvable nếu inversion chẵn
    return inv % 2 == 0


# ======================================================
# MODEL-BASED AGENT
# BFS + Loop Detection
# ======================================================
class PuzzleAgent:

    def __init__(self):
        self.visited = set()

    def get_neighbors(self, state):

        neighbors = []

        # tìm ô trống
        for i in range(SIZE):
            for j in range(SIZE):
                if state[i][j] == 0:
                    r, c = i, j

        directions = {
            "up": (1, 0),
            "down": (-1, 0),
            "left": (0, 1),
            "right": (0, -1)
        }

        for action, (dr, dc) in directions.items():

            nr = r + dr
            nc = c + dc

            if 0 <= nr < SIZE and 0 <= nc < SIZE:

                board = [list(row) for row in state]

                board[r][c], board[nr][nc] = \
                    board[nr][nc], board[r][c]

                neighbors.append((
                    tuple(tuple(row) for row in board),
                    action
                ))

        return neighbors

    def solve(self, initial_state):

        # ==================================
        # UNSOLVABLE DETECTION
        # ==================================
        if not is_solvable(initial_state):
            return None

        queue = deque()

        queue.append((initial_state, []))

        self.visited.clear()
        self.visited.add(initial_state)

        while queue:

            current_state, path = queue.popleft()

            # Goal test
            if current_state == GOAL:
                return path

            for neighbor, action in self.get_neighbors(current_state):

                # LOOP DETECTION
                if neighbor not in self.visited:

                    self.visited.add(neighbor)

                    queue.append((
                        neighbor,
                        path + [action]
                    ))

        return None


# ======================================================
# GUI
# ======================================================
class PuzzleGUI:

    def __init__(self, root):

        self.root = root
        self.root.title("8 Puzzle - Model Based Agent")

        self.model = PuzzleModel()
        self.agent = PuzzleAgent()

        self.buttons = []

        self.frame = tk.Frame(root)
        self.frame.pack(pady=10)

        # tạo board
        for i in range(SIZE):

            row = []

            for j in range(SIZE):

                btn = tk.Button(
                    self.frame,
                    width=6,
                    height=3,
                    font=("Arial", 24),
                    command=lambda r=i, c=j:
                    self.click_tile(r, c)
                )

                btn.grid(row=i, column=j, padx=2, pady=2)

                row.append(btn)

            self.buttons.append(row)

        # =========================
        # CONTROL BUTTONS
        # =========================
        control = tk.Frame(root)
        control.pack(pady=10)

        tk.Button(
            control,
            text="↑",
            width=5,
            command=lambda: self.move("up")
        ).grid(row=0, column=1)

        tk.Button(
            control,
            text="←",
            width=5,
            command=lambda: self.move("left")
        ).grid(row=1, column=0)

        tk.Button(
            control,
            text="↓",
            width=5,
            command=lambda: self.move("down")
        ).grid(row=1, column=1)

        tk.Button(
            control,
            text="→",
            width=5,
            command=lambda: self.move("right")
        ).grid(row=1, column=2)

        # =========================
        # FUNCTION BUTTONS
        # =========================
        tk.Button(
            root,
            text="Shuffle",
            font=("Arial", 14),
            command=self.shuffle
        ).pack(pady=5)

        tk.Button(
            root,
            text="Auto Solve (AI)",
            font=("Arial", 14),
            command=self.auto_solve
        ).pack(pady=5)

        self.status = tk.Label(
            root,
            text="",
            font=("Arial", 14)
        )

        self.status.pack()

        # keyboard
        root.bind("<Up>", lambda e: self.move("up"))
        root.bind("<Down>", lambda e: self.move("down"))
        root.bind("<Left>", lambda e: self.move("left"))
        root.bind("<Right>", lambda e: self.move("right"))

        self.update_ui()

    # ======================================
    # UPDATE UI
    # ======================================
    def update_ui(self):

        for i in range(SIZE):
            for j in range(SIZE):

                value = self.model.state[i][j]

                if value == 0:
                    self.buttons[i][j].config(
                        text="",
                        bg="lightgray"
                    )
                else:
                    self.buttons[i][j].config(
                        text=str(value),
                        bg="white"
                    )

        if self.model.get_state_tuple() == GOAL:
            self.status.config(text="Solved!")
        else:
            self.status.config(text="")

    # ======================================
    # MOVE
    # ======================================
    def move(self, direction):

        self.model.move(direction)

        self.update_ui()

    # ======================================
    # CLICK TILE
    # ======================================
    def click_tile(self, r, c):

        er, ec = self.model.find_empty()

        if abs(r - er) + abs(c - ec) == 1:

            self.model.state[er][ec], \
                self.model.state[r][c] = \
                self.model.state[r][c], \
                self.model.state[er][ec]

            self.update_ui()

    # ======================================
    # SHUFFLE
    # ======================================
    def shuffle(self):

        self.model.shuffle()

        self.update_ui()

    # ======================================
    # AI SOLVER
    # ======================================
    def auto_solve(self):

        current_state = self.model.get_state_tuple()

        # kiểm tra vô nghiệm
        if not is_solvable(current_state):

            self.status.config(
                text="UNSOLVABLE STATE DETECTED"
            )

            return

        path = self.agent.solve(current_state)

        if path is None:

            self.status.config(
                text="LOOP / NO SOLUTION"
            )

            return

        self.animate_solution(path)

    # ======================================
    # ANIMATION
    # ======================================
    def animate_solution(self, path):

        if not path:
            return

        move = path.pop(0)

        self.model.move(move)

        self.update_ui()

        self.root.after(
            300,
            lambda: self.animate_solution(path)
        )


# ======================================================
# MAIN
# ======================================================
root = tk.Tk()

app = PuzzleGUI(root)

root.mainloop()